In [1]:
from datetime import datetime


class PaperTradingEngine:
    """
    Paper trading engine for the Bitcoin trading system.

    This engine simulates:
        - BTC purchases
        - BTC sales
        - Trading fees
        - Portfolio value
        - Realized P/L
        - Unrealized P/L
        - Trade history
        - Active trades

    IMPORTANT:
        This class NEVER sends orders to Coinbase.
        It is strictly for simulation/testing.
    """

    def __init__(
        self,
        initial_cash_usd=10000,
        trading_fee_pct=0.006
    ):
        """
        Parameters
        ----------
        initial_cash_usd : float
            Starting paper-trading balance.

        trading_fee_pct : float
            Trading fee as a decimal.

            Example:
                0.006 = 0.6%
                0.001 = 0.1%
        """

        self.initial_cash_usd = float(initial_cash_usd)
        self.cash_usd = float(initial_cash_usd)

        self.btc_quantity = 0.0

        self.trading_fee_pct = float(trading_fee_pct)

        # Portfolio accounting
        self.realized_pnl = 0.0

        # Active positions
        self.active_positions = []

        # Complete trade history
        self.trade_history = []

        # Last known BTC price
        self.last_price = None

    # =========================================================
    # MARKET PRICE
    # =========================================================

    def update_price(self, btc_price):
        """
        Update the latest BTC price.
        """

        btc_price = float(btc_price)

        if btc_price <= 0:
            raise ValueError(
                "BTC price must be greater than zero."
            )

        self.last_price = btc_price

    # =========================================================
    # PORTFOLIO VALUE
    # =========================================================

    def get_btc_value(self, btc_price=None):
        """
        Calculate current BTC position value.
        """

        if btc_price is None:
            btc_price = self.last_price

        if btc_price is None:
            raise ValueError(
                "BTC price is required."
            )

        return self.btc_quantity * float(btc_price)

    def get_portfolio_value(self, btc_price=None):
        """
        Calculate total portfolio value.

        Portfolio =
            Cash + BTC value
        """

        return (
            self.cash_usd
            + self.get_btc_value(btc_price)
        )

    # =========================================================
    # BUY
    # =========================================================

    def buy_btc(
        self,
        usd_amount,
        btc_price,
        strategy="UNKNOWN",
        reason=""
    ):
        """
        Simulate a BTC purchase.

        Parameters
        ----------
        usd_amount : float
            USD amount to spend.

        btc_price : float
            BTC price.

        strategy : str
            Strategy responsible for the trade.

        reason : str
            Explanation for the trade.

        Returns
        -------
        dict
            Trade information.
        """

        usd_amount = float(usd_amount)
        btc_price = float(btc_price)

        if usd_amount <= 0:
            raise ValueError(
                "USD amount must be greater than zero."
            )

        if btc_price <= 0:
            raise ValueError(
                "BTC price must be greater than zero."
            )

        if usd_amount > self.cash_usd:
            raise ValueError(
                f"Insufficient paper cash. "
                f"Available: ${self.cash_usd:,.2f}"
            )

        # Calculate trading fee
        fee = usd_amount * self.trading_fee_pct

        # Total cash deducted
        total_cost = usd_amount + fee

        if total_cost > self.cash_usd:
            raise ValueError(
                "Insufficient cash after trading fee."
            )

        # BTC received
        btc_bought = usd_amount / btc_price

        # Update portfolio
        self.cash_usd -= total_cost
        self.btc_quantity += btc_bought

        self.last_price = btc_price

        # Record position
        position = {
            "entry_time": datetime.now().isoformat(),
            "entry_price": btc_price,
            "btc_quantity": btc_bought,
            "usd_cost": usd_amount,
            "fee": fee,
            "strategy": strategy,
            "reason": reason
        }

        self.active_positions.append(position)

        # Record trade
        trade = {
            "timestamp": datetime.now().isoformat(),
            "action": "BUY",
            "strategy": strategy,
            "btc_price": btc_price,
            "btc_quantity": btc_bought,
            "usd_amount": usd_amount,
            "fee": fee,
            "reason": reason
        }

        self.trade_history.append(trade)

        return trade

    # =========================================================
    # SELL
    # =========================================================

    def sell_btc(
        self,
        btc_quantity,
        btc_price,
        strategy="UNKNOWN",
        reason=""
    ):
        """
        Simulate selling BTC.

        FIFO accounting is used to calculate realized P/L.
        """

        btc_quantity = float(btc_quantity)
        btc_price = float(btc_price)

        if btc_quantity <= 0:
            raise ValueError(
                "BTC quantity must be greater than zero."
            )

        if btc_quantity > self.btc_quantity:
            raise ValueError(
                f"Cannot sell {btc_quantity:.8f} BTC. "
                f"Only {self.btc_quantity:.8f} BTC available."
            )

        self.last_price = btc_price

        # Gross sale value
        gross_value = btc_quantity * btc_price

        # Trading fee
        fee = gross_value * self.trading_fee_pct

        # Net amount received
        net_value = gross_value - fee

        # -----------------------------------------------------
        # Calculate realized P/L using FIFO
        # -----------------------------------------------------

        remaining_to_sell = btc_quantity
        cost_basis = 0.0

        while remaining_to_sell > 0 and self.active_positions:

            position = self.active_positions[0]

            position_btc = position["btc_quantity"]

            quantity_from_position = min(
                remaining_to_sell,
                position_btc
            )

            cost_per_btc = (
                position["usd_cost"]
                / position_btc
            )

            cost_basis += (
                quantity_from_position
                * cost_per_btc
            )

            position["btc_quantity"] -= (
                quantity_from_position
            )

            remaining_to_sell -= (
                quantity_from_position
            )

            if position["btc_quantity"] <= 0:
                self.active_positions.pop(0)

        realized_pnl = (
            net_value - cost_basis
        )

        self.realized_pnl += realized_pnl

        # Update portfolio
        self.btc_quantity -= btc_quantity
        self.cash_usd += net_value

        # Record trade
        trade = {
            "timestamp": datetime.now().isoformat(),
            "action": "SELL",
            "strategy": strategy,
            "btc_price": btc_price,
            "btc_quantity": btc_quantity,
            "gross_value": gross_value,
            "fee": fee,
            "net_value": net_value,
            "cost_basis": cost_basis,
            "realized_pnl": realized_pnl,
            "reason": reason
        }

        self.trade_history.append(trade)

        return trade

    # =========================================================
    # UNREALIZED P/L
    # =========================================================

    def get_unrealized_pnl(self, btc_price=None):
        """
        Calculate unrealized P/L on currently held BTC.

        Unrealized P/L =
            Current BTC value - estimated cost basis
        """

        if btc_price is None:
            btc_price = self.last_price

        if btc_price is None:
            raise ValueError(
                "BTC price is required."
            )

        total_cost = 0.0

        for position in self.active_positions:

            remaining_btc = position["btc_quantity"]

            if remaining_btc > 0:

                cost_per_btc = (
                    position["usd_cost"]
                    / (
                        position["btc_quantity"]
                        if position["btc_quantity"] > 0
                        else 1
                    )
                )

                total_cost += (
                    remaining_btc
                    * cost_per_btc
                )

        current_value = (
            self.btc_quantity
            * btc_price
        )

        return current_value - total_cost

    # =========================================================
    # TOTAL P/L
    # =========================================================

    def get_total_pnl(self, btc_price=None):
        """
        Calculate total portfolio P/L relative to
        the initial cash balance.
        """

        portfolio_value = self.get_portfolio_value(
            btc_price
        )

        return (
            portfolio_value
            - self.initial_cash_usd
        )

    # =========================================================
    # RETURN %
    # =========================================================

    def get_return_pct(self, btc_price=None):
        """
        Calculate portfolio return percentage.
        """

        total_pnl = self.get_total_pnl(
            btc_price
        )

        return (
            total_pnl
            / self.initial_cash_usd
        ) * 100

    # =========================================================
    # PORTFOLIO SUMMARY
    # =========================================================

    def get_portfolio_summary(self, btc_price=None):
        """
        Return complete portfolio information.
        """

        if btc_price is None:
            btc_price = self.last_price

        portfolio_value = self.get_portfolio_value(
            btc_price
        )

        total_pnl = self.get_total_pnl(
            btc_price
        )

        return {
            "timestamp": datetime.now().isoformat(),
            "cash_usd": self.cash_usd,
            "btc_quantity": self.btc_quantity,
            "btc_price": btc_price,
            "btc_value": self.get_btc_value(
                btc_price
            ),
            "portfolio_value": portfolio_value,
            "initial_value": self.initial_cash_usd,
            "total_pnl": total_pnl,
            "return_pct": (
                total_pnl
                / self.initial_cash_usd
            ) * 100,
            "realized_pnl": self.realized_pnl,
            "unrealized_pnl": self.get_unrealized_pnl(
                btc_price
            ),
            "active_positions": len(
                self.active_positions
            ),
            "total_trades": len(
                self.trade_history
            )
        }

    # =========================================================
    # TRADE HISTORY
    # =========================================================

    def get_trade_history(self):
        """
        Return all paper trades.
        """

        return self.trade_history

    # =========================================================
    # RESET
    # =========================================================

    def reset(self):
        """
        Reset the paper trading account.
        """

        self.cash_usd = self.initial_cash_usd
        self.btc_quantity = 0.0

        self.realized_pnl = 0.0

        self.active_positions = []
        self.trade_history = []

        self.last_price = None

In [2]:
engine = PaperTradingEngine(
    initial_cash_usd=10000,
    trading_fee_pct=0.006
)

In [3]:
trade = engine.buy_btc(
    usd_amount=500,
    btc_price=100000,
    strategy="DCA",
    reason="BTC dropped 3%"
)

print(trade)

{'timestamp': '2026-08-31T01:28:16.960139', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 100000.0, 'btc_quantity': 0.005, 'usd_amount': 500.0, 'fee': 3.0, 'reason': 'BTC dropped 3%'}


In [4]:
summary = engine.get_portfolio_summary(
    btc_price=100000
)

print(summary)

{'timestamp': '2026-08-31T01:28:35.999168', 'cash_usd': 9497.0, 'btc_quantity': 0.005, 'btc_price': 100000, 'btc_value': 500.0, 'portfolio_value': 9997.0, 'initial_value': 10000.0, 'total_pnl': -3.0, 'return_pct': -0.03, 'realized_pnl': 0.0, 'unrealized_pnl': 0.0, 'active_positions': 1, 'total_trades': 1}


In [5]:
trade = engine.buy_btc(
    usd_amount=500,
    btc_price=97000,
    strategy="DCA",
    reason="BTC dropped another 3%"
)

print(trade)

{'timestamp': '2026-08-31T01:28:52.634101', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 97000.0, 'btc_quantity': 0.005154639175257732, 'usd_amount': 500.0, 'fee': 3.0, 'reason': 'BTC dropped another 3%'}


In [6]:
print(
    engine.get_portfolio_summary(
        btc_price=97000
    )
)

{'timestamp': '2026-08-31T01:29:08.628762', 'cash_usd': 8994.0, 'btc_quantity': 0.010154639175257732, 'btc_price': 97000, 'btc_value': 985.0, 'portfolio_value': 9979.0, 'initial_value': 10000.0, 'total_pnl': -21.0, 'return_pct': -0.21, 'realized_pnl': 0.0, 'unrealized_pnl': -15.0, 'active_positions': 2, 'total_trades': 2}


In [7]:
btc_to_sell = engine.btc_quantity / 2

trade = engine.sell_btc(
    btc_quantity=btc_to_sell,
    btc_price=100000,
    strategy="ATR",
    reason="ATR stop-loss"
)

print(trade)

{'timestamp': '2026-08-31T01:29:21.917372', 'action': 'SELL', 'strategy': 'ATR', 'btc_price': 100000.0, 'btc_quantity': 0.005077319587628866, 'gross_value': 507.7319587628866, 'fee': 3.04639175257732, 'net_value': 504.6855670103093, 'cost_basis': 507.5, 'realized_pnl': -2.814432989690715, 'reason': 'ATR stop-loss'}


In [8]:
history = engine.get_trade_history()

for trade in history:
    print(trade)

{'timestamp': '2026-08-31T01:28:16.960139', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 100000.0, 'btc_quantity': 0.005, 'usd_amount': 500.0, 'fee': 3.0, 'reason': 'BTC dropped 3%'}
{'timestamp': '2026-08-31T01:28:52.634101', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 97000.0, 'btc_quantity': 0.005154639175257732, 'usd_amount': 500.0, 'fee': 3.0, 'reason': 'BTC dropped another 3%'}
{'timestamp': '2026-08-31T01:29:21.917372', 'action': 'SELL', 'strategy': 'ATR', 'btc_price': 100000.0, 'btc_quantity': 0.005077319587628866, 'gross_value': 507.7319587628866, 'fee': 3.04639175257732, 'net_value': 504.6855670103093, 'cost_basis': 507.5, 'realized_pnl': -2.814432989690715, 'reason': 'ATR stop-loss'}


In [17]:
class RiskManager:
    """
    Portfolio-level risk management for the Bitcoin trading agent.

    The RiskManager does NOT execute trades.

    It evaluates proposed trades and determines whether
    they should be approved or rejected.

    Main protections:
        - Maximum portfolio budget
        - Maximum individual trade size
        - Maximum active position
        - Global portfolio drawdown protection
        - Maximum number of active trades
        - Paper trading mode
    """

    def __init__(
        self,
        total_budget_usd=10000,
        max_trade_usd=1000,
        max_position_usd=5000,
        global_stop_loss_pct=25.0,
        max_active_trades=3,
        paper_trading=True
    ):
        """
        Initialize the RiskManager.

        Parameters
        ----------
        total_budget_usd : float
            Maximum amount of capital allocated to the system.

        max_trade_usd : float
            Maximum USD amount allowed for one trade.

        max_position_usd : float
            Maximum total BTC position value.

        global_stop_loss_pct : float
            Maximum portfolio drawdown before all trading
            activity is paused.

        max_active_trades : int
            Maximum number of simultaneous active trades.

        paper_trading : bool
            If True, the system must not place real orders.
        """

        self.total_budget_usd = float(total_budget_usd)
        self.max_trade_usd = float(max_trade_usd)
        self.max_position_usd = float(max_position_usd)
        self.global_stop_loss_pct = float(global_stop_loss_pct)

        self.max_active_trades = int(max_active_trades)

        self.paper_trading = bool(paper_trading)

        # Track simulated/current portfolio state
        self.cash_usd = self.total_budget_usd
        self.btc_quantity = 0.0

        # Track portfolio starting value
        self.initial_portfolio_value = self.total_budget_usd

        # Number of currently active trades
        self.active_trades = 0

        # Global trading pause
        self.trading_paused = False

    # ---------------------------------------------------------
    # Portfolio calculations
    # ---------------------------------------------------------

    def get_position_value(self, btc_price):
        """
        Calculate current BTC position value.
        """

        btc_price = float(btc_price)

        return self.btc_quantity * btc_price

    def get_portfolio_value(self, btc_price):
        """
        Calculate total portfolio value.

        Portfolio =
            Cash + BTC position
        """

        return (
            self.cash_usd
            + self.get_position_value(btc_price)
        )

    def get_drawdown_pct(self, btc_price):
        """
        Calculate portfolio drawdown from initial value.
        """

        portfolio_value = self.get_portfolio_value(
            btc_price
        )

        drawdown_pct = (
            (portfolio_value - self.initial_portfolio_value)
            / self.initial_portfolio_value
        ) * 100

        return drawdown_pct

    # ---------------------------------------------------------
    # Global portfolio safeguard
    # ---------------------------------------------------------

    def check_global_stop(self, btc_price):
        """
        Check whether the portfolio has exceeded the
        maximum allowed drawdown.
        """

        drawdown_pct = self.get_drawdown_pct(btc_price)

        if drawdown_pct <= -self.global_stop_loss_pct:

            self.trading_paused = True

            return {
                "approved": False,
                "reason": (
                    f"Global portfolio stop triggered. "
                    f"Drawdown: {drawdown_pct:.2f}%"
                ),
                "drawdown_pct": drawdown_pct
            }

        return {
            "approved": True,
            "reason": "Global portfolio stop not triggered",
            "drawdown_pct": drawdown_pct
        }

    # ---------------------------------------------------------
    # Trade size validation
    # ---------------------------------------------------------

    def validate_trade_size(self, trade_amount_usd):
        """
        Check whether an individual trade is within limits.
        """

        trade_amount_usd = float(trade_amount_usd)

        if trade_amount_usd <= 0:

            return {
                "approved": False,
                "reason": "Trade amount must be greater than zero"
            }

        if trade_amount_usd > self.max_trade_usd:

            return {
                "approved": False,
                "reason": (
                    f"Trade amount ${trade_amount_usd:,.2f} "
                    f"exceeds maximum trade size "
                    f"${self.max_trade_usd:,.2f}"
                )
            }

        return {
            "approved": True,
            "reason": "Trade size is within limits"
        }

    # ---------------------------------------------------------
    # Cash validation
    # ---------------------------------------------------------

    def validate_cash(self, trade_amount_usd):
        """
        Check whether sufficient cash is available.
        """

        trade_amount_usd = float(trade_amount_usd)

        if trade_amount_usd > self.cash_usd:

            return {
                "approved": False,
                "reason": (
                    f"Insufficient cash. "
                    f"Available: ${self.cash_usd:,.2f}, "
                    f"Requested: ${trade_amount_usd:,.2f}"
                )
            }

        return {
            "approved": True,
            "reason": "Sufficient cash available"
        }

    # ---------------------------------------------------------
    # Position-size validation
    # ---------------------------------------------------------

    def validate_position_size(
        self,
        trade_amount_usd,
        btc_price
    ):
        """
        Make sure the new position won't exceed the
        maximum allowed BTC position.
        """

        current_position = self.get_position_value(
            btc_price
        )

        new_position = (
            current_position + trade_amount_usd
        )

        if new_position > self.max_position_usd:

            return {
                "approved": False,
                "reason": (
                    f"Trade would exceed maximum position. "
                    f"Current: ${current_position:,.2f}, "
                    f"New: ${new_position:,.2f}, "
                    f"Maximum: ${self.max_position_usd:,.2f}"
                )
            }

        return {
            "approved": True,
            "reason": "Position size is within limits"
        }

    # ---------------------------------------------------------
    # Active trade validation
    # ---------------------------------------------------------

    def validate_active_trades(self):
        """
        Check whether another active trade can be opened.
        """

        if self.active_trades >= self.max_active_trades:

            return {
                "approved": False,
                "reason": (
                    f"Maximum active trades reached: "
                    f"{self.max_active_trades}"
                )
            }

        return {
            "approved": True,
            "reason": "Active trade limit not reached"
        }

    # ---------------------------------------------------------
    # Complete trade validation
    # ---------------------------------------------------------

    def approve_trade(
        self,
        trade_amount_usd,
        btc_price,
        trade_type="UNKNOWN"
    ):
        """
        Perform all risk checks for a proposed trade.

        Returns
        -------
        dict
            Approval/rejection decision.
        """

        trade_amount_usd = float(trade_amount_usd)
        btc_price = float(btc_price)

        # -----------------------------------------------------
        # Check global pause
        # -----------------------------------------------------

        if self.trading_paused:

            return {
                "approved": False,
                "trade_type": trade_type,
                "reason": "Trading is globally paused"
            }

        # -----------------------------------------------------
        # Global portfolio stop
        # -----------------------------------------------------

        global_check = self.check_global_stop(
            btc_price
        )

        if not global_check["approved"]:

            return {
                "approved": False,
                "trade_type": trade_type,
                "reason": global_check["reason"]
            }

        # -----------------------------------------------------
        # Trade size
        # -----------------------------------------------------

        size_check = self.validate_trade_size(
            trade_amount_usd
        )

        if not size_check["approved"]:

            return {
                "approved": False,
                "trade_type": trade_type,
                "reason": size_check["reason"]
            }

        # -----------------------------------------------------
        # Cash
        # -----------------------------------------------------

        cash_check = self.validate_cash(
            trade_amount_usd
        )

        if not cash_check["approved"]:

            return {
                "approved": False,
                "trade_type": trade_type,
                "reason": cash_check["reason"]
            }

        # -----------------------------------------------------
        # Position
        # -----------------------------------------------------

        position_check = self.validate_position_size(
            trade_amount_usd,
            btc_price
        )

        if not position_check["approved"]:

            return {
                "approved": False,
                "trade_type": trade_type,
                "reason": position_check["reason"]
            }

        # -----------------------------------------------------
        # Active trade limit
        # -----------------------------------------------------

        if trade_type != "DCA":

            active_check = self.validate_active_trades()

            if not active_check["approved"]:

                return {
                    "approved": False,
                    "trade_type": trade_type,
                    "reason": active_check["reason"]
                }

        # -----------------------------------------------------
        # All checks passed
        # -----------------------------------------------------

        return {
            "approved": True,
            "trade_type": trade_type,
            "trade_amount_usd": trade_amount_usd,
            "btc_price": btc_price,
            "reason": "All risk checks passed"
        }

    # ---------------------------------------------------------
    # Simulate BUY
    # ---------------------------------------------------------

    def record_buy(
        self,
        trade_amount_usd,
        btc_price,
        trade_type="DCA"
    ):
        """
        Record a simulated BTC purchase.

        This method updates the internal portfolio state.
        It does NOT send an order to Coinbase.
        """

        trade_amount_usd = float(trade_amount_usd)
        btc_price = float(btc_price)

        if trade_amount_usd > self.cash_usd:

            raise ValueError(
                "Cannot record purchase: insufficient cash."
            )

        btc_bought = (
            trade_amount_usd / btc_price
        )

        self.cash_usd -= trade_amount_usd
        self.btc_quantity += btc_bought

        if trade_type != "DCA":
            self.active_trades += 1

        return {
            "trade_type": trade_type,
            "usd_spent": trade_amount_usd,
            "btc_bought": btc_bought,
            "btc_price": btc_price,
            "remaining_cash": self.cash_usd,
            "btc_quantity": self.btc_quantity
        }

    # ---------------------------------------------------------
    # Simulate SELL
    # ---------------------------------------------------------

    def record_sell(
        self,
        btc_quantity,
        btc_price,
        trade_type="SWING"
    ):
        """
        Record a simulated BTC sale.

        This method does NOT send a real Coinbase order.
        """

        btc_quantity = float(btc_quantity)
        btc_price = float(btc_price)

        if btc_quantity <= 0:

            raise ValueError(
                "BTC quantity must be greater than zero."
            )

        if btc_quantity > self.btc_quantity:

            raise ValueError(
                "Cannot sell more BTC than currently held."
            )

        usd_received = (
            btc_quantity * btc_price
        )

        self.btc_quantity -= btc_quantity
        self.cash_usd += usd_received

        if trade_type != "DCA" and self.active_trades > 0:
            self.active_trades -= 1

        return {
            "trade_type": trade_type,
            "btc_sold": btc_quantity,
            "btc_price": btc_price,
            "usd_received": usd_received,
            "remaining_cash": self.cash_usd,
            "btc_quantity": self.btc_quantity
        }

    # ---------------------------------------------------------
    # Pause / resume
    # ---------------------------------------------------------

    def pause_trading(self):
        """
        Manually pause all trading.
        """

        self.trading_paused = True

    def resume_trading(self):
        """
        Resume trading after a manual pause.
        """

        self.trading_paused = False

    # ---------------------------------------------------------
    # Portfolio summary
    # ---------------------------------------------------------

    def get_portfolio_summary(self, btc_price):
        """
        Return a summary of the current portfolio.
        """

        portfolio_value = self.get_portfolio_value(
            btc_price
        )

        drawdown_pct = self.get_drawdown_pct(
            btc_price
        )

        return {
            "cash_usd": self.cash_usd,
            "btc_quantity": self.btc_quantity,
            "btc_price": btc_price,
            "btc_position_value": self.get_position_value(
                btc_price
            ),
            "portfolio_value": portfolio_value,
            "drawdown_pct": drawdown_pct,
            "active_trades": self.active_trades,
            "trading_paused": self.trading_paused,
            "paper_trading": self.paper_trading
        }

print("risk_manager.py created successfully!")

risk_manager.py created successfully!


In [14]:
class DCAStrategy:
    """
    Dollar-Cost Averaging strategy.

    The strategy buys a fixed USD amount of BTC when:
    1. DCA is enabled
    2. The BTC price has fallen by the configured percentage
       since the previous DCA purchase
    """

    def __init__(
        self,
        buy_amount_usd=500,
        drop_pct=3.0,
        enabled=True
    ):
        self.buy_amount_usd = float(buy_amount_usd)
        self.drop_pct = float(drop_pct)
        self.enabled = enabled

        # Price at which the previous DCA purchase occurred
        self.last_buy_price = None

    def should_buy(self, current_price):
        """
        Determine whether a DCA purchase should occur.

        Returns
        -------
        dict
            Decision information.
        """

        if not self.enabled:
            return {
                "action": "HOLD",
                "reason": "DCA strategy is disabled"
            }

        current_price = float(current_price)

        # No previous purchase price
        if self.last_buy_price is None:
            return {
                "action": "BUY",
                "amount_usd": self.buy_amount_usd,
                "price": current_price,
                "reason": "Initial DCA purchase"
            }

        price_change_pct = (
            (current_price - self.last_buy_price)
            / self.last_buy_price
        ) * 100

        required_drop = -self.drop_pct

        if price_change_pct <= required_drop:

            return {
                "action": "BUY",
                "amount_usd": self.buy_amount_usd,
                "price": current_price,
                "price_change_pct": price_change_pct,
                "reason": (
                    f"BTC dropped {abs(price_change_pct):.2f}% "
                    f"since last DCA purchase"
                )
            }

        return {
            "action": "HOLD",
            "price": current_price,
            "price_change_pct": price_change_pct,
            "reason": (
                f"BTC has not dropped {self.drop_pct}% "
                "since last DCA purchase"
            )
        }

    def record_buy(self, price):
        """
        Record the price of the most recent DCA purchase.
        """

        self.last_buy_price = float(price)

In [18]:
dca = DCAStrategy(
    buy_amount_usd=500,
    drop_pct=3.0,
    enabled=True
)

price = 100000

decision = dca.should_buy(price)

print(decision)

{'action': 'BUY', 'amount_usd': 500.0, 'price': 100000.0, 'reason': 'Initial DCA purchase'}


In [20]:
risk = RiskManager(
    total_budget_usd=10000,
    max_trade_usd=1000,
    max_position_usd=5000,
    global_stop_loss_pct=25,
    max_active_trades=3,
    paper_trading=True
)

In [21]:
if decision["action"] == "BUY":

    approval = risk.approve_trade(
        trade_amount_usd=decision["amount_usd"],
        btc_price=price,
        trade_type="DCA"
    )

    print(approval)

{'approved': True, 'trade_type': 'DCA', 'trade_amount_usd': 500.0, 'btc_price': 100000.0, 'reason': 'All risk checks passed'}


In [22]:

class ATRStrategy:
    """
    ATR-based strategy for active/swing trades.

    This strategy uses ATR (Average True Range) to determine
    a dynamic stop-loss level.

    Stop Loss:
        Entry Price - (ATR × multiplier)

    This class DOES NOT execute real trades.
    It only generates trading decisions and manages
    the stop-loss information for an active trade.
    """

    def __init__(
        self,
        atr_multiplier=1.5,
        enabled=True
    ):
        """
        Initialize the ATR strategy.

        Parameters
        ----------
        atr_multiplier : float
            Multiplier applied to ATR when calculating
            the stop-loss.

        enabled : bool
            Whether the ATR strategy is enabled.
        """

        self.atr_multiplier = float(atr_multiplier)
        self.enabled = enabled

        # Information about the current active trade
        self.entry_price = None
        self.entry_atr = None
        self.stop_price = None

    def calculate_stop_loss(self, entry_price, atr):
        """
        Calculate the stop-loss price.

        Formula:
            Stop = Entry Price - (ATR × multiplier)

        Parameters
        ----------
        entry_price : float
            Price at which the trade was entered.

        atr : float
            Current ATR value.

        Returns
        -------
        float
            Stop-loss price.
        """

        entry_price = float(entry_price)
        atr = float(atr)

        if entry_price <= 0:
            raise ValueError("Entry price must be greater than zero.")

        if atr < 0:
            raise ValueError("ATR cannot be negative.")

        stop_price = (
            entry_price
            - (self.atr_multiplier * atr)
        )

        return stop_price

    def generate_entry_signal(
        self,
        current_price,
        atr,
        rsi=None,
        macd=None,
        macd_signal=None,
        volume_ratio=None
    ):
        """
        Generate a potential swing-trade entry signal.

        This is intentionally conservative.

        The strategy looks for optional confirmation from:
            - RSI
            - MACD
            - Volume

        Parameters
        ----------
        current_price : float
            Current BTC price.

        atr : float
            Current ATR(14).

        rsi : float, optional
            RSI value.

        macd : float, optional
            MACD value.

        macd_signal : float, optional
            MACD signal value.

        volume_ratio : float, optional
            Current volume divided by average volume.

        Returns
        -------
        dict
            Entry decision.
        """

        current_price = float(current_price)
        atr = float(atr)

        if not self.enabled:
            return {
                "action": "HOLD",
                "price": current_price,
                "reason": "ATR strategy is disabled"
            }

        if current_price <= 0:
            raise ValueError("Current price must be greater than zero.")

        if atr <= 0:
            return {
                "action": "HOLD",
                "price": current_price,
                "reason": "Invalid or unavailable ATR"
            }

        # -------------------------------------------------
        # Confirmation conditions
        # -------------------------------------------------

        confirmations = 0
        reasons = []

        # RSI confirmation
        if rsi is not None:
            if 40 <= rsi <= 70:
                confirmations += 1
                reasons.append("RSI supports momentum")

        # MACD confirmation
        if macd is not None and macd_signal is not None:
            if macd > macd_signal:
                confirmations += 1
                reasons.append("MACD bullish")

        # Volume confirmation
        if volume_ratio is not None:
            if volume_ratio >= 1.2:
                confirmations += 1
                reasons.append("volume breakout")

        # -------------------------------------------------
        # Entry rule
        #
        # Require at least two confirmations when
        # indicators are available.
        # -------------------------------------------------

        available_confirmations = sum([
            rsi is not None,
            macd is not None and macd_signal is not None,
            volume_ratio is not None
        ])

        if available_confirmations >= 2 and confirmations >= 2:

            stop_price = self.calculate_stop_loss(
                current_price,
                atr
            )

            return {
                "action": "BUY",
                "price": current_price,
                "atr": atr,
                "stop_price": stop_price,
                "atr_multiplier": self.atr_multiplier,
                "confirmations": confirmations,
                "reason": "; ".join(reasons)
            }

        return {
            "action": "HOLD",
            "price": current_price,
            "atr": atr,
            "confirmations": confirmations,
            "reason": "Insufficient bullish confirmations"
        }

    def open_trade(self, entry_price, atr):
        """
        Record an active trade.

        This does not execute a Coinbase order.
        """

        self.entry_price = float(entry_price)
        self.entry_atr = float(atr)

        self.stop_price = self.calculate_stop_loss(
            self.entry_price,
            self.entry_atr
        )

    def check_stop_loss(self, current_price):
        """
        Check whether the active trade has reached
        its stop-loss.

        Returns
        -------
        dict
            Stop-loss decision.
        """

        current_price = float(current_price)

        if self.entry_price is None:
            return {
                "action": "NO_TRADE",
                "price": current_price,
                "reason": "No active trade"
            }

        if current_price <= self.stop_price:

            loss_pct = (
                (current_price - self.entry_price)
                / self.entry_price
            ) * 100

            return {
                "action": "SELL",
                "price": current_price,
                "entry_price": self.entry_price,
                "stop_price": self.stop_price,
                "loss_pct": loss_pct,
                "reason": "ATR stop-loss triggered"
            }

        return {
            "action": "HOLD",
            "price": current_price,
            "entry_price": self.entry_price,
            "stop_price": self.stop_price,
            "reason": "Stop-loss not triggered"
        }

    def close_trade(self):
        """
        Clear the active trade information.
        """

        self.entry_price = None
        self.entry_atr = None
        self.stop_price = None

print("atr_strategy.py created successfully!")

atr_strategy.py created successfully!


In [23]:
atr_strategy = ATRStrategy(
    atr_multiplier=1.5,
    enabled=True
)

In [24]:
decision = atr_strategy.generate_entry_signal(
    current_price=62500,
    atr=1000,
    rsi=60,
    macd=100,
    macd_signal=50,
    volume_ratio=1.5
)

print(decision)

{'action': 'BUY', 'price': 62500.0, 'atr': 1000.0, 'stop_price': 61000.0, 'atr_multiplier': 1.5, 'confirmations': 3, 'reason': 'RSI supports momentum; MACD bullish; volume breakout'}


In [25]:
if decision["action"] == "BUY":

    approval = risk.approve_trade(
        trade_amount_usd=1000,
        btc_price=62500,
        trade_type="SWING"
    )

    print(approval)

{'approved': True, 'trade_type': 'SWING', 'trade_amount_usd': 1000.0, 'btc_price': 62500.0, 'reason': 'All risk checks passed'}


In [26]:
if approval["approved"]:

    trade = engine.buy_btc(
        usd_amount=1000,
        btc_price=62500,
        strategy="ATR",
        reason=decision["reason"]
    )

    atr_strategy.open_trade(
        entry_price=62500,
        atr=1000
    )

    print(trade)

{'timestamp': '2026-08-31T01:38:48.401491', 'action': 'BUY', 'strategy': 'ATR', 'btc_price': 62500.0, 'btc_quantity': 0.016, 'usd_amount': 1000.0, 'fee': 6.0, 'reason': 'RSI supports momentum; MACD bullish; volume breakout'}
